# YOLO26 vs RF-DETR bake-off, and Moondream 2 vs 3.1 on bumpers

One notebook, one dataset, one GPU, one metric. Run the cells top to bottom.

**Budget (Colab Pro, 200 compute units/month).** Rates are approximate; Colab shows the exact rate under *Runtime → Change runtime type*.

| GPU | ~units/hour | use it for |
|---|---|---|
| T4 | 1.5–2 | the dry run (`DRY_RUN = True`), Moondream 2 |
| **L4** | 4–5 | **the real run** and Moondream 3.1 |
| A100 | 11–13 | only if the L4 runs out of memory at 960 px |

A real run is about 3–8 h per model on an L4, roughly 40–75 units for both plus the bumper test.

**Before you start**, put this in Google Drive under `MyDrive/frc-bakeoff/`:

```
frc-bakeoff/
  dataset-scout.zip     # zip of the dataset folder: dataset.yaml, images/{train,val}, labels/{train,val}
  videos/               # optional: match videos for the bumper test (e.g. qm7.mp4)
```

Everything the run writes (weights, checkpoints, results) goes to `frc-bakeoff/runs/` in Drive, so a disconnect loses nothing: re-run the cells and training resumes.

**Do the dry run first on a T4** (`DRY_RUN = True`: 1 epoch on a few images, ~10 min). It catches a bad zip or path before you spend L4 hours. Then switch the runtime to L4, set `DRY_RUN = False` and run everything again.

## 1. Settings

In [ ]:
DRY_RUN = True            # True: 1 epoch on a few images to test the setup. False: the real run.

DRIVE_DIR   = "/content/drive/MyDrive/frc-bakeoff"
DATASET_ZIP = "dataset-scout.zip"     # in DRIVE_DIR
BRANCH      = "claude/practical-einstein-bgrb3r"   # switch to "main" once this work is merged

IMGSZ       = 960         # fuel is ~17x12 px; 960 is what train.py uses. RF-DETR needs a multiple of 32.
EPOCHS      = 50
YOLO_MODEL  = "yolo26s.pt"
RFDETR_SIZE = "small"     # nano / small / medium
BATCH       = 8           # lower to 4 if either model runs out of GPU memory

# Bumper test on a real match (optional). Team numbers from TBA's match page.
BUMPER_VIDEO = ""         # e.g. "videos/qm7.mp4" (relative to DRIVE_DIR); "" skips it
RED_TEAMS    = ""         # e.g. "1058,611,8013"
BLUE_TEAMS   = ""

## 2. GPU check

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 / L4 / A100"
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{name}: compute capability {major}.{minor}, {mem:.0f} GB")
CAN_RUN_MD31 = major >= 8     # Moondream 3.1's Photon engine needs Ampere or newer (L4, A100: yes; T4: no)
print("Moondream 3.1 can run here:", CAN_RUN_MD31)
if not DRY_RUN and "T4" in name:
    print("!! Real run on a T4: it will work but slowly. An L4 is about 3x faster for ~2.5x the units.")

## 3. Drive, code, packages

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
assert os.path.exists(f"{DRIVE_DIR}/{DATASET_ZIP}"), f"put {DATASET_ZIP} in {DRIVE_DIR} first"

In [ ]:
%cd /content
!rm -rf frc && git clone -q --depth 1 -b {BRANCH} https://github.com/ShadowOfTheVOID/YOLOv26-FRC-Model frc
%cd /content/frc
# runs/ lives in Drive: weights and checkpoints survive a disconnect
!rm -rf runs && ln -s "{DRIVE_DIR}/runs" runs
!pip -q install "ultralytics>=8.4" rfdetr

## 4. Dataset
Copied from Drive to the local disk: reading thousands of small files straight from Drive is slow enough to starve the GPU.

In [ ]:
import glob, shutil, random
!rm -rf /content/ds && mkdir -p /content/ds && unzip -q "{DRIVE_DIR}/{DATASET_ZIP}" -d /content/ds
found = glob.glob("/content/ds/**/dataset.yaml", recursive=True)
assert found, "no dataset.yaml inside the zip"
DATA = found[0]
root = pathlib.Path(DATA).parent
for split in ("train", "val"):
    n = len(list((root / "images" / split).glob("*")))
    print(split, n, "images")

if DRY_RUN:   # a few images per split, so the dry run takes minutes
    small = pathlib.Path("/content/ds_dry")
    shutil.rmtree(small, ignore_errors=True)
    for split, k in (("train", 16), ("val", 8)):
        imgs = sorted((root / "images" / split).glob("*"))
        random.Random(0).shuffle(imgs)
        for kind in ("images", "labels"):
            (small / kind / split).mkdir(parents=True)
        for p in imgs[:k]:
            shutil.copy(p, small / "images" / split)
            lab = root / "labels" / split / (p.stem + ".txt")
            if lab.exists():
                shutil.copy(lab, small / "labels" / split)
    shutil.copy(DATA, small / "dataset.yaml")
    DATA = str(small / "dataset.yaml")
print("training on", DATA)

In [ ]:
TAG = "dry" if DRY_RUN else "bake"
EP = 1 if DRY_RUN else EPOCHS
YOLO_NAME, RF_NAME = f"yolo_{TAG}", f"rfdetr_{TAG}"

## 5. Train YOLO26
If this run was cut off, re-running the cell resumes from `last.pt` in Drive.

In [ ]:
run = pathlib.Path(f"runs/{YOLO_NAME}")
last = run / "weights" / "last.pt"
if (run / "DONE").exists():
    print("already trained:", run / "weights" / "best.pt", "(delete DONE in Drive to train again)")
elif last.exists():
    print("resuming", last)
    !python train/train.py --model {last} --resume --data {DATA} --name {YOLO_NAME} && touch {run}/DONE
else:
    !python train/train.py --data {DATA} --model {YOLO_MODEL} --imgsz {IMGSZ} --epochs {EP} --batch {BATCH} --workers 4 --name {YOLO_NAME} && touch {run}/DONE

## 6. Train RF-DETR
Same data, same resolution. Re-running the cell resumes from `checkpoint.pth` in Drive.

In [ ]:
run = pathlib.Path(f"runs/{RF_NAME}")
ckpt = run / "checkpoint.pth"
if (run / "DONE").exists():
    print("already trained:", run / "checkpoint_best_total.pth", "(delete DONE in Drive to train again)")
else:
    resume = f"--resume {ckpt}" if ckpt.exists() else ""
    if resume: print("resuming", ckpt)
    !python train/train_rfdetr.py --data {DATA} --size {RFDETR_SIZE} --imgsz {IMGSZ} --epochs {EP} --batch {max(1, BATCH // 2)} --grad-accum 4 --workers 4 --name {RF_NAME} {resume} && touch {run}/DONE

## 7. Compare on the same val images with the same metric

`AP50sm` (boxes under 24 px) is the column that decides fuel. All numbers are agreement with the auto-labels, so look at disagreements by eye before believing a small gap.

In [ ]:
out = f"runs/compare_{TAG}.txt"
!python train/compare_detectors.py --data {DATA} --imgsz {IMGSZ} \
    --yolo runs/{YOLO_NAME}/weights/best.pt \
    --rfdetr runs/{RF_NAME}/checkpoint_best_total.pth --rfdetr-size {RFDETR_SIZE} | tee {out}
print("saved to Drive:", out)

## 8. Bumpers: Moondream 2 vs 3.1 on the same crops

The size test from `experiments/bumper_moondream.py`: one public 2024 field photo (Wikimedia Commons), each robot shrunk until its numerals are 35 → 7 px tall. On a 1080p wide broadcast shot they are about 10 px. Moondream 2 on CPU got 360 right down to 7 px, read 3268 as "32 69" at 14 px, and made up "131" for a half-hidden bumper. The question is whether 3.1 does better at 10–14 px.

In [ ]:
!pip -q install "transformers>=4.51,<5"   # Moondream 2 was measured with 4.57
import urllib.request
PHOTO = "/content/crescendo.jpg"
URL = "https://upload.wikimedia.org/wikipedia/commons/6/60/FIRST_Robotics_Competition_%28FRC%29_2024_Crescendo_Playing_Field.jpg"
if os.path.exists(f"{DRIVE_DIR}/crescendo.jpg"):
    shutil.copy(f"{DRIVE_DIR}/crescendo.jpg", PHOTO)
elif not os.path.exists(PHOTO):
    try:
        req = urllib.request.Request(URL, headers={"User-Agent": "frc-bakeoff-notebook/0.1"})
        open(PHOTO, "wb").write(urllib.request.urlopen(req, timeout=60).read())
    except Exception as e:
        raise SystemExit(f"Could not download the test photo ({e}). Download it in a browser from\n{URL}\n"
                         f"and put it in Drive as {DRIVE_DIR}/crescendo.jpg, then re-run this cell.")
# boxes are for the 3840 px wide version the table was measured on
from PIL import Image
im = Image.open(PHOTO)
if im.width != 3840:
    im.resize((3840, round(im.height * 3840 / im.width)), Image.LANCZOS).save(PHOTO)
BOXES = "--box 2130,716,2407,926=360 --box 2010,780,2165,895=3268 --box 3331,768,3601,933="
!python experiments/bumper_moondream.py --moondream 2 --scale-test {PHOTO} {BOXES} | tee runs/bumpers_md2.txt

In [ ]:
if CAN_RUN_MD31:
    !pip -q install moondream
    !python experiments/bumper_moondream.py --moondream 3.1 --scale-test {PHOTO} {BOXES} | tee runs/bumpers_md31.txt
else:
    print("Moondream 3.1 needs an Ampere or newer GPU (L4 / A100). Switch runtime to run this cell.")

## 9. Bumpers on a real match (optional)

Reads every robot box once a second with the better detector and writes a CSV to Drive. Answers are only accepted when they name a team on that alliance (exact or one digit off). Open the CSV and check a sample by eye: right team, wrong team, or blank.

In [ ]:
if BUMPER_VIDEO and not DRY_RUN:
    weights = f"runs/{YOLO_NAME}/weights/best.pt"   # or runs/{RF_NAME}/checkpoint_best_total.pth if RF-DETR won
    version = "3.1" if CAN_RUN_MD31 else "2"
    !python experiments/bumper_moondream.py --moondream {version} --video "{DRIVE_DIR}/{BUMPER_VIDEO}" \
        --weights {weights} --red {RED_TEAMS} --blue {BLUE_TEAMS} --every 1.0 --out runs/bumper_reads.csv
else:
    print("skipped: set BUMPER_VIDEO (and DRY_RUN = False)")

## 10. Stop paying
An idle GPU session keeps using compute units. Everything worth keeping is already in Drive.

In [ ]:
from google.colab import runtime
runtime.unassign()